In [ ]:
import subprocess, sys, os, json, textwrap

PKGS = [
    "apache-ossie @ git+https://github.com/apache/ossie.git#subdirectory=python",
    "jsonschema>=4.26.0", "pyyaml>=6.0", "sqlglot>=30.12.0", "duckdb>=1.0", "pandas",
]
print("Installing dependencies (apache-ossie is not on PyPI yet — installing from git)...")
_pip = [sys.executable, "-m", "pip", "install", "-q"]
if subprocess.run(_pip + PKGS).returncode:
    subprocess.run(_pip + ["--break-system-packages", *PKGS], check=True)

REPO = "/content/ossie" if os.path.isdir("/content") else "./ossie"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    "https://github.com/apache/ossie.git", REPO], check=True)
print(f"Repo checked out at {REPO}\n")

import yaml, duckdb, sqlglot
from sqlglot import exp
from jsonschema import Draft202012Validator
from collections import deque, defaultdict
from ossie import (
    OSIDocument, OSISemanticModel, OSIDataset, OSIField, OSIMetric,
    OSIRelationship, OSIExpression, OSIDialectExpression, OSIDimension,
    OSIAIContextObject, OSICustomExtension, OSIDialect, OSIDataType, OSIVendor,
)

SCHEMA_PATH = f"{REPO}/core-spec/osi-schema.json"
SCHEMA = json.load(open(SCHEMA_PATH))
VALIDATOR = Draft202012Validator(SCHEMA)
SPEC_VERSION = SCHEMA["properties"]["version"]["const"]

def rule(title):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

rule("PART 1 — The Ossie core spec, read from its own JSON Schema")

defs = SCHEMA["$defs"]
print(f"spec version pinned by schema : {SPEC_VERSION}")
print(f"root required keys            : {SCHEMA['required']}")
print(f"root additionalProperties     : {SCHEMA['additionalProperties']}  <- strict!")
print(f"dialects                      : {defs['Dialect']['enum']}")
print(f"datatypes                     : {defs['DataType']['enum']}")
print("\nobject contracts:")
for name in ("SemanticModel", "Dataset", "Field", "Relationship", "Metric", "CustomExtension"):
    d = defs[name]
    print(f"  {name:16s} required={d.get('required', [])}")
    print(f"  {'':16s} optional={[k for k in d['properties'] if k not in d.get('required', [])]}")

rule("PART 2 — Validating the shipped TPC-DS reference model")

tpcds = yaml.safe_load(open(f"{REPO}/examples/tpcds_semantic_model.yaml"))
errors = sorted(VALIDATOR.iter_errors(tpcds), key=lambda e: list(e.absolute_path))
print(f"schema errors: {len(errors)}")
m = tpcds["semantic_model"][0]
print(f"model '{m['name']}': {len(m['datasets'])} datasets, "
      f"{len(m.get('relationships', []))} relationships, {len(m.get('metrics', []))} metrics")
print("metrics:", ", ".join(x["name"] for x in m["metrics"]))
print("\nOne metric, verbatim:")
print(textwrap.indent(yaml.dump(m["metrics"][0], sort_keys=False), "  "))

In [ ]:
rule("PART 3 — Authoring a retail model with the Pydantic SDK")

def sql(expr: str, dialect: OSIDialect = OSIDialect.ANSI_SQL) -> OSIExpression:
    """Shorthand for a single-dialect expression object."""
    return OSIExpression(dialects=[OSIDialectExpression(dialect=dialect, expression=expr)])

def field(name, expr, datatype=None, desc=None, is_time=None, synonyms=None, label=None):
    return OSIField(
        name=name, expression=sql(expr), label=label, description=desc, datatype=datatype,
        dimension=OSIDimension(is_time=is_time) if is_time is not None else None,
        ai_context=OSIAIContextObject(synonyms=tuple(synonyms)) if synonyms else None,
    )

customers = OSIDataset(
    name="customers", source="retail.customers", primary_key=["customer_id"],
    description="One row per registered customer.",
    ai_context=OSIAIContextObject(
        instructions="Customer master. Join through orders to reach revenue.",
        synonyms=("buyers", "accounts", "shoppers")),
    fields=[
        field("customer_id", "customer_id", OSIDataType.INTEGER, "Surrogate key"),
        field("customer_name", "full_name", OSIDataType.STRING, "Display name", synonyms=["name"]),
        field("region", "region", OSIDataType.STRING, "Sales region", is_time=False,
              synonyms=["territory", "area", "geo"]),
        field("segment", "segment", OSIDataType.STRING, "Loyalty tier", is_time=False,
              synonyms=["tier", "loyalty level"]),
        field("signup_date", "signup_date", OSIDataType.DATE, "Account creation date", is_time=True),
        field("signup_year", "EXTRACT(YEAR FROM signup_date)", OSIDataType.INTEGER,
              "Cohort year", is_time=True, synonyms=["cohort"]),
    ],
)

products = OSIDataset(
    name="products", source="retail.products", primary_key=["product_id"],
    description="Sellable SKUs.",
    ai_context=OSIAIContextObject(synonyms=("items", "SKUs", "catalog")),
    fields=[
        field("product_id", "product_id", OSIDataType.INTEGER, "Surrogate key"),
        field("product_name", "product_name", OSIDataType.STRING, "SKU name"),
        field("category", "category", OSIDataType.STRING, "Merch category", is_time=False,
              synonyms=["department", "product type"]),
        field("list_price", "list_price", OSIDataType.DECIMAL, "Catalog price"),
    ],
)

orders = OSIDataset(
    name="orders", source="retail.orders", primary_key=["order_id"],
    description="Order headers; grain = one order.",
    ai_context=OSIAIContextObject(
        instructions="Header grain. Never SUM line-level money here — use order_items.",
        synonyms=("purchases", "transactions")),
    fields=[
        field("order_id", "order_id", OSIDataType.INTEGER, "Surrogate key"),
        field("customer_id", "customer_id", OSIDataType.INTEGER, "FK -> customers"),
        field("order_date", "order_date", OSIDataType.DATE, "Order placement date", is_time=True,
              synonyms=["date", "purchase date"]),
        field("order_month", "DATE_TRUNC('month', order_date)", OSIDataType.DATE,
              "Month bucket", is_time=True, synonyms=["month"]),
        field("channel", "channel", OSIDataType.STRING, "Sales channel", is_time=False,
              synonyms=["source", "medium"]),
        field("status", "status", OSIDataType.STRING, "Fulfilment status", is_time=False),
    ],
)

order_items = OSIDataset(
    name="order_items", source="retail.order_items",
    primary_key=["order_id", "line_number"],
    unique_keys=[["order_id", "line_number"]],
    description="Order line items; the money grain of this model.",
    ai_context=OSIAIContextObject(
        instructions="Line grain. All revenue metrics are defined here.",
        synonyms=("line items", "order lines")),
    fields=[
        field("order_id", "order_id", OSIDataType.INTEGER, "FK -> orders"),
        field("line_number", "line_number", OSIDataType.INTEGER, "Line ordinal"),
        field("product_id", "product_id", OSIDataType.INTEGER, "FK -> products"),
        field("quantity", "quantity", OSIDataType.INTEGER, "Units on the line"),
        field("unit_price", "unit_price", OSIDataType.DECIMAL, "Price paid per unit"),
        field("discount_pct", "discount_pct", OSIDataType.FLOAT, "0..1 discount fraction"),
    ],
    custom_extensions=[OSICustomExtension(
        vendor_name=OSIVendor.DBT.value,
        data=json.dumps({"materialized": "incremental", "unique_key": "order_line_sk"}))],
)

GROSS = "SUM(order_items.quantity * order_items.unit_price)"
NET = "SUM(order_items.quantity * order_items.unit_price * (1 - order_items.discount_pct))"

def metric(name, expr, desc, datatype=OSIDataType.DECIMAL, synonyms=(), examples=()):
    return OSIMetric(
        name=name, expression=sql(expr), description=desc, datatype=datatype,
        ai_context=OSIAIContextObject(synonyms=tuple(synonyms), examples=tuple(examples)))

retail = OSISemanticModel(
    name="retail_analytics",
    description="Vendor-neutral retail semantic model: orders, lines, customers, products.",
    ai_context=OSIAIContextObject(
        instructions=("Answer commercial questions with these metrics only. "
                      "Never hand-roll aggregations over raw columns."),
        examples=("What was net revenue by region last quarter?",
                  "Which category has the highest average order value?")),
    datasets=[customers, products, orders, order_items],
    relationships=[
        OSIRelationship(name="items_to_orders", **{"from": "order_items"}, to="orders",
                        from_columns=["order_id"], to_columns=["order_id"]),
        OSIRelationship(name="items_to_products", **{"from": "order_items"}, to="products",
                        from_columns=["product_id"], to_columns=["product_id"]),
        OSIRelationship(name="orders_to_customers", **{"from": "orders"}, to="customers",
                        from_columns=["customer_id"], to_columns=["customer_id"],
                        ai_context=OSIAIContextObject(synonyms=("who placed the order",))),
    ],
    metrics=[
        metric("gross_revenue", GROSS, "Revenue before discounts",
               synonyms=["gross sales", "gmv", "revenue before discount"]),
        metric("net_revenue", NET, "Revenue after line discounts",
               synonyms=["revenue", "net sales", "sales", "turnover"],
               examples=["net revenue by month", "net revenue by category"]),
        metric("discount_amount", f"{GROSS} - ({NET})", "Absolute discount given",
               synonyms=["markdown", "discounts"]),
        metric("units_sold", "SUM(order_items.quantity)", "Total units shipped",
               OSIDataType.INTEGER, ["volume", "quantity", "units"]),
        metric("order_count", "COUNT(DISTINCT orders.order_id)", "Distinct orders",
               OSIDataType.INTEGER, ["orders", "number of orders", "transactions"]),
        metric("customer_count", "COUNT(DISTINCT customers.customer_id)", "Distinct buyers",
               OSIDataType.INTEGER, ["buyers", "unique customers", "active customers"]),
        metric("avg_order_value", f"{NET} / NULLIF(COUNT(DISTINCT orders.order_id), 0)",
               "Net revenue per order", synonyms=["aov", "average order value", "basket size"]),
        metric("revenue_per_customer", f"{NET} / NULLIF(COUNT(DISTINCT customers.customer_id), 0)",
               "Net revenue per buyer", synonyms=["arpu", "ltv proxy", "revenue per buyer"]),
    ],
)

doc = OSIDocument(version=SPEC_VERSION, semantic_model=[retail])
print(f"built '{retail.name}': {len(retail.datasets)} datasets, "
      f"{sum(len(d.fields or []) for d in retail.datasets)} fields, "
      f"{len(retail.relationships)} relationships, {len(retail.metrics)} metrics")
print("models are frozen (immutable):", end=" ")
try:
    retail.datasets[0].name = "nope"
except Exception as e:
    print(type(e).__name__)

rule("PART 4 — Serialization and a real interoperability gotcha")

payload = json.loads(doc.to_osi_json())
print("schema errors on our document:", len(list(VALIDATOR.iter_errors(payload))))

yaml_text = doc.to_osi_yaml()
open("retail_model.yaml", "w").write(yaml_text)
print("\nfirst 22 lines of retail_model.yaml:")
print(textwrap.indent("\n".join(yaml_text.splitlines()[:22]), "  "))

drifting = OSIDocument(version=SPEC_VERSION, dialects=[OSIDialect.ANSI_SQL],
                       semantic_model=[retail])
drift_errs = [e.message for e in VALIDATOR.iter_errors(json.loads(drifting.to_osi_json()))]
print("\nSDK-legal but schema-illegal document ->", drift_errs)

reparsed = OSIDocument.model_validate(yaml.safe_load(yaml_text))
print("round-trip stable:", reparsed.to_osi_yaml() == yaml_text)

In [ ]:
rule("PART 5 — Governance linter")

def columns_of(expression: str):
    """Return {(table, column)} referenced by a SQL expression, via sqlglot."""
    try:
        tree = sqlglot.parse_one(expression)
    except Exception:
        try:
            tree = sqlglot.parse_one(f"SELECT {expression}")
        except Exception:
            return None
    return {(c.table or "", c.name) for c in tree.find_all(exp.Column)}

def lint(document: dict):
    problems, warnings = [], []
    for model in document["semantic_model"]:
        ds_by_name = {d["name"]: d for d in model["datasets"]}
        cols = {n: {f["name"] for f in d.get("fields", [])} for n, d in ds_by_name.items()}

        def dupes(names):
            seen, out = set(), []
            for n in names:
                (out.append(n) if n in seen else None); seen.add(n)
            return out

        for dup in dupes(list(ds_by_name)):
            problems.append(f"duplicate dataset '{dup}'")
        for name, d in ds_by_name.items():
            for dup in dupes([f["name"] for f in d.get("fields", [])]):
                problems.append(f"duplicate field '{name}.{dup}'")
            for pk in d.get("primary_key", []):
                if pk not in {f["expression"]["dialects"][0]["expression"]
                              for f in d.get("fields", [])}:
                    warnings.append(f"primary key column '{name}.{pk}' has no exposed field")

        for rel in model.get("relationships", []):
            for side in ("from", "to"):
                if rel[side] not in ds_by_name:
                    problems.append(f"relationship '{rel['name']}' -> unknown dataset '{rel[side]}'")
            if len(rel["from_columns"]) != len(rel["to_columns"]):
                problems.append(f"relationship '{rel['name']}' has mismatched key arity")

        for met in model.get("metrics", []):
            for de in met["expression"]["dialects"]:
                refs = columns_of(de["expression"])
                if refs is None:
                    problems.append(f"metric '{met['name']}' has unparseable {de['dialect']} SQL")
                    continue
                for tbl, col in refs:
                    if not tbl:
                        problems.append(f"metric '{met['name']}' uses unqualified column '{col}'")
                    elif tbl not in ds_by_name:
                        problems.append(f"metric '{met['name']}' references unknown dataset '{tbl}'")
                    elif col not in cols[tbl]:
                        problems.append(f"metric '{met['name']}' references unknown field '{tbl}.{col}'")

        adj = defaultdict(set)
        for rel in model.get("relationships", []):
            adj[rel["from"]].add(rel["to"]); adj[rel["to"]].add(rel["from"])
        if ds_by_name:
            start = next(iter(ds_by_name)); seen, q = {start}, deque([start])
            while q:
                for nb in adj[q.popleft()]:
                    if nb not in seen:
                        seen.add(nb); q.append(nb)
            for orphan in set(ds_by_name) - seen:
                problems.append(f"dataset '{orphan}' is unreachable in the join graph")

        described = sum(1 for d in model["datasets"] for f in d.get("fields", []) if f.get("description"))
        total = sum(len(d.get("fields", [])) for d in model["datasets"])
        ai = sum(1 for m in model.get("metrics", []) if m.get("ai_context"))
        warnings.append(f"field description coverage {described}/{total}"
                        f" | metric ai_context coverage {ai}/{len(model.get('metrics', []))}")
    return problems, warnings

probs, warns = lint(payload)
print(f"clean model -> {len(probs)} problems")
for w in warns: print("  note:", w)

broken = json.loads(json.dumps(payload))
bm = broken["semantic_model"][0]
bm["metrics"].append({"name": "bad_metric",
                      "expression": {"dialects": [{"dialect": "ANSI_SQL",
                                                   "expression": "SUM(orders.gross_margin)"}]}})
bm["relationships"].append({"name": "ghost", "from": "orders", "to": "warehouses",
                            "from_columns": ["w_id"], "to_columns": ["id"]})
bm["datasets"].append({"name": "shipments", "source": "retail.shipments"})
print(f"\nsabotaged model -> {len(lint(broken)[0])} problems (schema says it is still valid: "
      f"{len(list(VALIDATOR.iter_errors(broken))) == 0})")
for p in lint(broken)[0]: print("  ✗", p)

rule("PART 6 — Join graph")

class SemanticGraph:
    def __init__(self, model: dict):
        self.model = model
        self.datasets = {d["name"]: d for d in model["datasets"]}
        self.metrics = {m["name"]: m for m in model.get("metrics", [])}
        self.fields = {(d["name"], f["name"]): f
                       for d in model["datasets"] for f in d.get("fields", [])}
        self.edges = defaultdict(list)
        for rel in model.get("relationships", []):
            self.edges[rel["from"]].append((rel["to"], rel, False))
            self.edges[rel["to"]].append((rel["from"], rel, True))

    def path(self, src, dst):
        """Shortest chain of relationships from src to dst (BFS)."""
        if src == dst: return []
        prev, q = {src: None}, deque([src])
        while q:
            node = q.popleft()
            for nb, rel, rev in self.edges[node]:
                if nb not in prev:
                    prev[nb] = (node, rel, rev); q.append(nb)
                    if nb == dst:
                        chain, cur = [], dst
                        while prev[cur]:
                            node_, rel_, rev_ = prev[cur]; chain.append((node_, cur, rel_, rev_)); cur = node_
                        return list(reversed(chain))
        raise ValueError(f"no join path between '{src}' and '{dst}'")

graph = SemanticGraph(retail.model_dump(by_alias=True, exclude_none=True, mode="json"))
print("edges:")
for rel in graph.model["relationships"]:
    print(f"  {rel['from']:12s} --{rel['name']:22s}--> {rel['to']}"
          f"  on ({', '.join(rel['from_columns'])}) = ({', '.join(rel['to_columns'])})")
chain = graph.path("order_items", "customers")
print("\nresolved join path order_items -> customers:",
      " -> ".join([chain[0][0]] + [step[1] for step in chain]))

rule("PART 7 — Compiling the semantic model into SQL and running it")

def qualify(expression: str, alias: str, dialect=None) -> str:
    """Bind an unqualified (dataset-scoped) field expression to a table alias."""
    tree = sqlglot.parse_one(expression, read=dialect)
    for col in tree.find_all(exp.Column):
        if not col.table:
            col.set("table", exp.to_identifier(alias))
    return tree.sql(dialect=dialect)

def expr_for(obj, dialect="ANSI_SQL"):
    dl = obj["expression"]["dialects"]
    return next((d["expression"] for d in dl if d["dialect"] == dialect), dl[0]["expression"])

class QueryCompiler:
    """metrics + dimensions -> a single SQL statement over the physical sources."""

    def __init__(self, graph: SemanticGraph, dialect="ANSI_SQL"):
        self.g, self.dialect = graph, dialect

    def _datasets_in(self, metric_names, dim_refs):
        needed = set()
        for name in metric_names:
            for tbl, _ in columns_of(expr_for(self.g.metrics[name], self.dialect)) or set():
                needed.add(tbl)
        needed |= {ref.split(".")[0] for ref in dim_refs}
        return needed

    def compile(self, metrics, dimensions=(), filters=(), order_by=None, limit=None):
        for m in metrics:
            if m not in self.g.metrics: raise KeyError(f"unknown metric '{m}'")
        for d in dimensions:
            if tuple(d.split(".")) not in self.g.fields: raise KeyError(f"unknown dimension '{d}'")

        needed = self._datasets_in(metrics, dimensions)
        root = max(needed, key=lambda n: len(self.g.edges[n]))
        joined, joins = {root}, []
        for target in sorted(needed - {root}):
            for left, right, rel, reversed_ in self.g.path(root, target):
                if right in joined: continue
                lcols, rcols = ((rel["from_columns"], rel["to_columns"]) if not reversed_
                                else (rel["to_columns"], rel["from_columns"]))
                on = " AND ".join(f"{left}.{a} = {right}.{b}" for a, b in zip(lcols, rcols))
                joins.append(f"  LEFT JOIN {self.g.datasets[right]['source']} AS {right} ON {on}")
                joined.add(right)

        select, group = [], []
        for ref in dimensions:
            ds, fname = ref.split(".")
            bound = qualify(expr_for(self.g.fields[(ds, fname)], self.dialect), ds)
            select.append(f"  {bound} AS {fname}"); group.append(bound)
        for name in metrics:
            select.append(f"  {expr_for(self.g.metrics[name], self.dialect)} AS {name}")

        parts = ["SELECT", ",\n".join(select),
                 f"FROM {self.g.datasets[root]['source']} AS {root}"]
        parts += joins
        if filters:
            parts.append("WHERE " + "\n  AND ".join(filters))
        if group:
            parts.append("GROUP BY " + ", ".join(group))
        parts.append(f"ORDER BY {order_by or (metrics[0] + ' DESC')}")
        if limit: parts.append(f"LIMIT {limit}")
        return "\n".join(parts)

con = duckdb.connect()
con.execute("CREATE SCHEMA retail")
con.execute("""
CREATE TABLE retail.customers AS
SELECT i AS customer_id,
       'Customer ' || i AS full_name,
       ['North','South','East','West'][(i % 4) + 1] AS region,
       ['Enterprise','SMB','Consumer'][(i % 3) + 1] AS segment,
       DATE '2022-01-01' + INTERVAL (i * 3) DAY AS signup_date
FROM range(1, 401) t(i)""")
con.execute("""
CREATE TABLE retail.products AS
SELECT i AS product_id,
       'SKU-' || lpad(i::VARCHAR, 4, '0') AS product_name,
       ['Apparel','Electronics','Home','Grocery','Toys'][(i % 5) + 1] AS category,
       round(5 + (i * 7 % 300) + 0.99, 2)::DECIMAL(10,2) AS list_price
FROM range(1, 121) t(i)""")
con.execute("""
CREATE TABLE retail.orders AS
SELECT i AS order_id,
       ((i * 37) % 400) + 1 AS customer_id,
       DATE '2024-01-01' + INTERVAL ((i * 11) % 540) DAY AS order_date,
       ['web','mobile','store','partner'][(i % 4) + 1] AS channel,
       CASE WHEN i % 23 = 0 THEN 'cancelled' ELSE 'complete' END AS status
FROM range(1, 6001) t(i)""")
con.execute("""
CREATE TABLE retail.order_items AS
SELECT o.order_id,
       l AS line_number,
       ((o.order_id * l * 13) % 120) + 1 AS product_id,
       ((o.order_id + l) % 5) + 1 AS quantity,
       round(4 + ((o.order_id * l) % 250) + 0.49, 2)::DECIMAL(10,2) AS unit_price,
       CASE WHEN (o.order_id + l) % 7 = 0 THEN 0.15 ELSE 0.0 END AS discount_pct
FROM retail.orders o, range(1, 4) t(l)
WHERE (o.order_id + l) % 3 <> 0""")
print("warehouse rows:", {t: con.execute(f"SELECT count(*) FROM retail.{t}").fetchone()[0]
                          for t in ("customers", "products", "orders", "order_items")})

compiler = QueryCompiler(graph)

q1 = compiler.compile(
    metrics=["net_revenue", "order_count", "avg_order_value"],
    dimensions=["customers.region", "products.category"],
    filters=["orders.status = 'complete'"],
    order_by="net_revenue DESC", limit=8)
print("\n--- generated SQL ---\n" + q1)
print("\n--- result ---")
print(con.execute(q1).df().to_string(index=False))

q2 = compiler.compile(metrics=["net_revenue", "units_sold", "customer_count"],
                      dimensions=["orders.order_month"], order_by="order_month", limit=6)
print("\n--- time series (note the computed order_month field gets auto-qualified) ---")
print(con.execute(q2).df().to_string(index=False))

q3 = compiler.compile(metrics=["revenue_per_customer", "discount_amount"],
                      dimensions=["customers.segment", "orders.channel"], limit=6)
print("\n--- cross-dataset ratio metrics ---")
print(con.execute(q3).df().to_string(index=False))

total_a = con.execute(compiler.compile(["net_revenue"])).fetchone()[0]
total_b = con.execute(compiler.compile(["net_revenue"], ["products.category"])).df()["net_revenue"].sum()
print(f"\nmetric consistency: ungrouped={total_a:,.2f}  regrouped-sum={total_b:,.2f}  "
      f"match={abs(float(total_a) - float(total_b)) < 0.01}")

In [ ]:
rule("PART 8 — Auto-generating dialect variants with sqlglot")

DIALECT_MAP = {"SNOWFLAKE": "snowflake", "BIGQUERY": "bigquery", "DATABRICKS": "databricks"}

def add_dialects(model: OSISemanticModel, targets=("SNOWFLAKE", "BIGQUERY", "DATABRICKS")):
    def widen(expression: OSIExpression) -> OSIExpression:
        base = next(d for d in expression.dialects if d.dialect == OSIDialect.ANSI_SQL)
        variants = list(expression.dialects)
        for target in targets:
            try:
                out = sqlglot.transpile(base.expression, read=None,
                                        write=DIALECT_MAP[target])[0]
            except Exception:
                continue
            variants.append(OSIDialectExpression(dialect=OSIDialect(target), expression=out))
        return OSIExpression(dialects=variants)

    data = model.model_dump(by_alias=True, exclude_none=True, mode="json")
    for d in data["datasets"]:
        for f in d.get("fields", []):
            f["expression"] = json.loads(widen(OSIExpression.model_validate(f["expression"])).model_dump_json())
    for m in data.get("metrics", []):
        m["expression"] = json.loads(widen(OSIExpression.model_validate(m["expression"])).model_dump_json())
    return OSISemanticModel.model_validate(data)

multi = add_dialects(retail)
multi_doc = OSIDocument(version=SPEC_VERSION, semantic_model=[multi])
print("schema errors after enrichment:", len(list(VALIDATOR.iter_errors(json.loads(multi_doc.to_osi_json())))))
sample = next(m for m in multi.metrics if m.name == "avg_order_value")
for de in sample.expression.dialects:
    print(f"  {de.dialect.value:11s} {de.expression}")
month = next(f for f in multi.datasets[2].fields if f.name == "order_month")
print("\n  computed field 'order_month' across dialects:")
for de in month.expression.dialects:
    print(f"  {de.dialect.value:11s} {de.expression}")

snow = QueryCompiler(SemanticGraph(multi.model_dump(by_alias=True, exclude_none=True, mode="json")),
                     dialect="SNOWFLAKE")
print("\nSnowflake-dialect query:\n" +
      snow.compile(["net_revenue"], ["orders.order_month"], limit=3))

rule("PART 9 — Introspecting DuckDB and generating an Ossie document")

TYPE_MAP = {"BIGINT": OSIDataType.INTEGER, "INTEGER": OSIDataType.INTEGER,
            "HUGEINT": OSIDataType.INTEGER, "VARCHAR": OSIDataType.STRING,
            "DOUBLE": OSIDataType.FLOAT, "FLOAT": OSIDataType.FLOAT,
            "BOOLEAN": OSIDataType.BOOLEAN, "DATE": OSIDataType.DATE,
            "TIMESTAMP": OSIDataType.DATE_TIME, "TIMESTAMP WITH TIME ZONE": OSIDataType.DATE_TIME_TZ}

def introspect(con, schema_name="retail", model_name="reverse_engineered"):
    cols = con.execute("""
        SELECT table_name, column_name, data_type
        FROM information_schema.columns WHERE table_schema = ?
        ORDER BY table_name, ordinal_position""", [schema_name]).fetchall()
    by_table = defaultdict(list)
    for t, c, dt in cols:
        by_table[t].append((c, dt.split("(")[0].upper()))

    datasets, relationships = [], []
    for table, columns in by_table.items():
        candidate = f"{table[:-1]}_id"
        pk = [candidate] if any(c == candidate for c, _ in columns) and \
             con.execute(f"SELECT count(*) = count(DISTINCT {candidate}) "
                         f"FROM {schema_name}.{table}").fetchone()[0] else None
        datasets.append(OSIDataset(
            name=table, source=f"{schema_name}.{table}", primary_key=pk,
            description=f"Auto-generated from {schema_name}.{table}",
            fields=[OSIField(name=c, expression=sql(c),
                             datatype=TYPE_MAP.get(dt, OSIDataType.OPAQUE),
                             dimension=OSIDimension(is_time=True) if TYPE_MAP.get(dt) in
                             (OSIDataType.DATE, OSIDataType.DATE_TIME) else None)
                    for c, dt in columns]))

    pk_owner = {d.primary_key[0]: d.name for d in datasets if d.primary_key}
    for d in datasets:
        for f in d.fields:
            owner = pk_owner.get(f.name)
            if owner and owner != d.name:
                relationships.append(OSIRelationship(
                    name=f"{d.name}_to_{owner}", **{"from": d.name}, to=owner,
                    from_columns=[f.name], to_columns=[f.name]))
    return OSIDocument(version=SPEC_VERSION, semantic_model=[OSISemanticModel(
        name=model_name, description=f"Reverse-engineered from DuckDB schema '{schema_name}'",
        datasets=datasets, relationships=relationships)])

auto = introspect(con)
print(f"discovered {len(auto.semantic_model[0].datasets)} datasets, "
      f"{len(auto.semantic_model[0].relationships)} inferred relationships")
for r in auto.semantic_model[0].relationships:
    print(f"  {r.from_dataset} -> {r.to} on {r.from_columns}")
print("schema errors:", len(list(VALIDATOR.iter_errors(json.loads(auto.to_osi_json())))))
print("lint problems:", lint(json.loads(auto.to_osi_json()))[0] or "none")

rule("PART 10 — Lossless round-trip through a foreign vendor format")

def to_vendor(model: OSISemanticModel) -> dict:
    """Ossie -> a fictional BI tool's flat format (loses grain, keeps definitions)."""
    return {
        "cube": model.name,
        "tables": [{"id": d.name, "sql_table": d.source,
                    "columns": [{"id": f.name, "sql": f.expression.dialects[0].expression,
                                 "type": (f.datatype.value if f.datatype else None)}
                                for f in (d.fields or [])]} for d in model.datasets],
        "measures": [{"id": m.name, "sql": m.expression.dialects[0].expression,
                      "title": m.description} for m in model.metrics or []],
        "joins": [{"id": r.name, "left": r.from_dataset, "right": r.to,
                   "on": list(zip(r.from_columns, r.to_columns))}
                  for r in model.relationships or []],
    }

def from_vendor(v: dict, carried: dict) -> OSISemanticModel:
    """Vendor -> Ossie, restoring what the vendor format cannot express."""
    extras = carried["datasets"]
    return OSISemanticModel(
        name=v["cube"], description=carried.get("description"),
        ai_context=carried.get("ai_context"),
        datasets=[OSIDataset(
            name=t["id"], source=t["sql_table"],
            primary_key=extras.get(t["id"], {}).get("primary_key"),
            unique_keys=extras.get(t["id"], {}).get("unique_keys"),
            fields=[OSIField(name=c["id"], expression=sql(c["sql"]),
                             datatype=OSIDataType(c["type"]) if c["type"] else None)
                    for c in t["columns"]]) for t in v["tables"]],
        relationships=[OSIRelationship(name=j["id"], **{"from": j["left"]}, to=j["right"],
                                       from_columns=[a for a, _ in j["on"]],
                                       to_columns=[b for _, b in j["on"]]) for j in v["joins"]],
        metrics=[OSIMetric(name=m["id"], expression=sql(m["sql"]), description=m["title"])
                 for m in v["measures"]],
    )

vendor_payload = to_vendor(retail)
sidecar = {"description": retail.description, "ai_context": retail.ai_context,
           "datasets": {d.name: {"primary_key": d.primary_key, "unique_keys": d.unique_keys}
                        for d in retail.datasets}}
recovered = from_vendor(vendor_payload, sidecar)
recovered_doc = OSIDocument(version=SPEC_VERSION, semantic_model=[recovered])
print("vendor format keys:", list(vendor_payload))
print("schema errors after round-trip:", len(list(VALIDATOR.iter_errors(json.loads(recovered_doc.to_osi_json())))))

def core_shape(m: OSISemanticModel):
    return {
        "datasets": {d.name: (d.source, tuple(d.primary_key or ()),
                              tuple((f.name, f.expression.dialects[0].expression) for f in (d.fields or [])))
                     for d in m.datasets},
        "metrics": {x.name: x.expression.dialects[0].expression for x in m.metrics},
        "joins": {r.name: (r.from_dataset, r.to, tuple(r.from_columns), tuple(r.to_columns))
                  for r in m.relationships},
    }
print("structural round-trip lossless:", core_shape(retail) == core_shape(recovered))
lost = [f.name for d in retail.datasets for f in (d.fields or []) if f.ai_context]
print(f"lost without a sidecar: ai_context on {len(lost)} fields, "
      f"{sum(len(d.custom_extensions or []) for d in retail.datasets)} custom_extensions"
      " -> this is exactly why the spec has custom_extensions")

In [1]:
rule("PART 11 — From ai_context to an answerable question router")

def context_pack(model: OSISemanticModel) -> str:
    """Compact, token-efficient briefing an LLM can be grounded on."""
    def ctx(o):
        if o is None: return ""
        if isinstance(o, str): return f" — {o}"
        bits = []
        if o.instructions: bits.append(o.instructions)
        if o.synonyms: bits.append("aka " + ", ".join(o.synonyms))
        return " — " + "; ".join(bits) if bits else ""
    lines = [f"# Semantic model: {model.name}", model.description or "", ctx(model.ai_context).strip(" —"),
             "\n## Metrics (use these; never aggregate raw columns)"]
    lines += [f"- {m.name}: {m.description}{ctx(m.ai_context)}" for m in model.metrics]
    lines.append("\n## Dimensions")
    for d in model.datasets:
        dims = [f for f in (d.fields or []) if f.dimension is not None]
        if dims:
            lines.append(f"- {d.name}{ctx(d.ai_context)}")
            lines += [f"    - {d.name}.{f.name} ({f.datatype.value if f.datatype else '?'})"
                      f"{' [time]' if f.is_time_dimension() else ''}{ctx(f.ai_context)}" for f in dims]
    lines.append("\n## Join graph")
    lines += [f"- {r.from_dataset} -> {r.to} on {list(zip(r.from_columns, r.to_columns))}"
              for r in model.relationships]
    return "\n".join(l for l in lines if l is not None)

pack = context_pack(retail)
print(pack)

def synonym_index(model):
    idx = defaultdict(list)
    for m in model.metrics:
        terms = [m.name.replace("_", " ")] + list(getattr(m.ai_context, "synonyms", ()) or ())
        for t in terms: idx[t.lower()].append(("metric", m.name))
    for d in model.datasets:
        for f in (d.fields or []):
            if f.dimension is None: continue
            terms = [f.name.replace("_", " ")] + list(getattr(f.ai_context, "synonyms", ()) or ())
            for t in terms: idx[t.lower()].append(("dimension", f"{d.name}.{f.name}"))
    return idx

INDEX = synonym_index(retail)

def ask(question: str, limit=5):
    q = question.lower()
    hits = {"metric": [], "dimension": []}
    for term, targets in sorted(INDEX.items(), key=lambda kv: -len(kv[0])):
        if term in q:
            for kind, name in targets:
                if name not in hits[kind] and not any(name in v for v in hits[kind]):
                    hits[kind].append(name)
    metrics = hits["metric"][:2] or ["net_revenue"]
    dims = hits["dimension"][:2]
    sql_text = compiler.compile(metrics, dims, limit=limit)
    print(f"\nQ: {question}\n   resolved -> metrics={metrics} dimensions={dims}")
    print(textwrap.indent(sql_text, "   | "))
    print(textwrap.indent(con.execute(sql_text).df().to_string(index=False), "   "))

ask("what is net revenue by region?")
ask("show me units and orders per category")
ask("average order value by channel")

rule("DONE")
print(f"""Artifacts produced in this session:
  retail_model.yaml          hand-authored, schema-valid Ossie document
  multi_doc                  same model widened to 4 SQL dialects
  auto                       model reverse-engineered from a live warehouse
  recovered_doc              model round-tripped through a foreign vendor format

Where to go next in {REPO}:
  core-spec/spec.md                     the normative spec ({SPEC_VERSION})
  core-spec/expression_language.md      expression semantics
  ontology/ontology.md                  the richer ontology track (see examples/flights.yaml)
  converters/                           dbt, Databricks, Snowflake, GoodData, Salesforce, ...
  validation/validate.py                the reference CLI validator
  compliance/README.md                  what "Ossie-compliant" means for a tool""")

Installing dependencies (apache-ossie is not on PyPI yet — installing from git)...
Repo checked out at /content/ossie


PART 1 — The Ossie core spec, read from its own JSON Schema
spec version pinned by schema : 0.2.0.dev0
root required keys            : ['version', 'semantic_model']
root additionalProperties     : False  <- strict!
dialects                      : ['ANSI_SQL', 'SNOWFLAKE', 'MDX', 'TABLEAU', 'DATABRICKS', 'MAQL', 'BIGQUERY']
datatypes                     : ['String', 'Integer', 'Decimal', 'Float', 'Boolean', 'Date', 'Time', 'DateTime', 'DateTimeTz', 'Opaque']

object contracts:
  SemanticModel    required=['name', 'datasets']
                   optional=['description', 'ai_context', 'relationships', 'metrics', 'custom_extensions']
  Dataset          required=['name', 'source']
                   optional=['primary_key', 'unique_keys', 'description', 'ai_context', 'fields', 'custom_extensions']
  Field            required=['name', 'expression']
                   optiona